# DATAMATCH 2026: Аналитический трек
## Методика комплексной оценки полевых игроков для ХК «Авангард»
### Воспроизводимый ноутбук генерации `test_ratings.csv`

**Цель:** Разработать и применить объективную методику оценки полезности полевых игроков (нападающих и защитников) для трансферных решений ХК «Авангард» на выборке 84 матчей сезона 2025/26.

**Архитектура методики:**
1. **Data Quality Audit & TOI Engine:** Выявление и разрешение 1067 конфликтов амплуа через заявку матча `lineup_*`, карантин 3 повреждённых матчей (G0072, G0190, G0352), коррекция эталона TOI (18 000 с на команду при 5v5).
2. **xG Model (LightGBM):** Оценка вероятности гола на 109 тыс. бросков с 5-Fold GroupKFold валидацией по матчам (ROC-AUC = 0.8096, Log Loss = 0.1683, Brier = 0.0441 — без паразитных утечек времени).
3. **Action Value / xThreat (xT):** Пространственная марковская сетка опасности передач, входов/выходов из зоны, отборов и потерь.
4. **Regularized Adjusted Plus-Minus (RAPM):** Изоляция индивидуального вклада игрока (атака + оборона за 60 минут) с контролем партнёров, соперников и численного преимущества методом Ridge L2 (Macdonald 2012, Nandakumar & Jensen 2019).
5. **Bootstrap Uncertainty:** 90% доверительные интервалы через блочный бутстрэп матчей (B=30).

In [ ]:
import os

import numpy as np
import pandas as pd
import src.toi as toi
import src.xg_model as xg
import src.action_value as av
import src.rapm as rapm
import src.evaluate as ev

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 200)
print("Все библиотеки и модули успешно импортированы.")

ModuleNotFoundError: No module named 'seaborn'

In [ ]:
candidates = [
    "DataMatch_Аналитический_трек1/data",
    "data",
    "../DataMatch_Аналитический_трек1/data",
    "../datamatch_hackathon/DataMatch_Аналитический_трек1/data",
    "../data",
    "."
]
data_dir = next((c for c in candidates if os.path.exists(os.path.join(c, "events.parquet"))), None)
if data_dir is None:
    raise FileNotFoundError(
        f"File (events.parquet) not found. Move it to one of the paths: {candidates}"
    )
print(f"Data path used: {data_dir}")

events_train = pd.read_parquet(f"{data_dir}/events.parquet")
games_train = pd.read_csv(f"{data_dir}/games.csv")
players_train = pd.read_csv(f"{data_dir}/players.csv")

events_test = pd.read_parquet(f"{data_dir}/test/events_test.parquet")
games_test = pd.read_csv(f"{data_dir}/test/games_test.csv")
players_new = pd.read_csv(f"{data_dir}/test/players_new.csv")
player_seasons_test = pd.read_csv(f"{data_dir}/test/player_seasons_test.csv")

# Изоляция 3 поврежденных матчей
events_train = events_train[~events_train.match_id.isin(toi.EXCLUDED_MATCHES)].copy()
games_train = games_train[~games_train.match_id.isin(toi.EXCLUDED_MATCHES)].copy()

all_players = pd.concat([players_train, players_new]).drop_duplicates(subset=['player_id'])

print(f"Обучающая выборка: {len(games_train)} матчей (3 в карантине), {len(events_train):,} событий")
print(f"Тестовая выборка: {len(games_test)} матчей «Авангарда», {len(events_test):,} событий")
print(f"Всего игроков: {len(all_players):,} уникальных профилей")

In [ ]:
# Восстановление смен и времени на льду 
print("Восстановление смен с разрешением ролей по заявке")
shifts_train = toi.extract_player_shifts(events_train, all_players, exclude_corrupted=True)
shifts_test = toi.extract_player_shifts(events_test, all_players, exclude_corrupted=False)

toi_train = toi.build_player_match_toi(shifts_train)
toi_test = toi.build_player_match_toi(shifts_test)

print(f"Извлечено смен: jбучение = {len(shifts_train):,}, тест = {len(shifts_test):,}")
print("\nПример рассчитанного TOI по игрокам за матч:")
print(toi_test.head(5))

In [ ]:
# Извлечение признаков и обучение модели xG (LightGBM)
print("Обучение модели Expected Goals (xG)")
shots_train = xg.extract_shot_features(events_train, games_train, all_players)
print(f"Бросков в обучающей выборке: {len(shots_train):,} (Голов: {shots_train['is_goal'].sum()}, {shots_train['is_goal'].mean()*100:.2f}%)")

xg_model, xg_metrics = xg.train_xg_model(shots_train, n_splits=5)
print(f"\nМетрики качества xG модели (5-Fold GroupKFold CV):")
print(f"ROC-AUC: {xg_metrics['roc_auc']:.4f}")
print(f"Log Loss: {xg_metrics['log_loss']:.4f}")
print(f"Brier Score: {xg_metrics['brier_score']:.4f}")

# Предсказание xG для тестовых бросков
shots_test = xg.extract_shot_features(events_test, games_test, all_players)
shots_test['xg'] = xg.predict_xg(xg_model, shots_test)
shots_train['xg'] = xg.predict_xg(xg_model, shots_train)

print(f"Тестовые броски оценены: {len(shots_test):,} бросков, сумма xG = {shots_test['xg'].sum():.1f} (факт голов: {shots_test['is_goal'].sum()})")

In [ ]:
# Пространственная сетка Expected Threat (xT) и оценка действий
print("Обучение сетки Expected Threat (xT)")
xt_model = av.ExpectedThreatModel(nx=12, ny=8)
xt_model.fit(events_train, shots_train['xg'])

print(f"Диапазон базовой угрозы на площадке: от {xt_model.grid.min():.4f} до {xt_model.grid.max():.4f}")

# Оценка небросковых действий в тесте (передачи, входы, отборы, потери)
actions_test = av.score_player_actions(events_test, xt_model)
print(f"Оценено действий в тесте: {len(actions_test):,}")
print(actions_test.groupby('action_category')['value'].describe())

In [ ]:
# Построение отрезков матчей (stints) и расчет RAPM
print("Построение отрезков постоянного состава (stints) для 84 матчей «Авангарда»...")

shot_xg_dict = {}
for r in shots_test[['match_id', 'period', 'clock_s', 'actor_team_id', 'xg']].itertuples(index=False):
    mid, per, clk, tid, val = r
    key = (mid, per)
    if key not in shot_xg_dict:
        shot_xg_dict[key] = []
    shot_xg_dict[key].append((clk, tid, val))

test_stints_by_match = {}
for _, g_row in games_test.iterrows():
    mid = g_row['match_id']
    ht = g_row['home_team_id']
    at = g_row['away_team_id']
    m_events = events_test[events_test['match_id'] == mid]
    m_shifts = shifts_test[shifts_test['match_id'] == mid]
    stints_list = toi.build_stints_for_match(m_events, m_shifts, ht, at)
    
    for s in stints_list:
        per = s['period']
        t0_c, t1_c = s['start_clock'], s['end_clock']
        cand_shots = shot_xg_dict.get((mid, per), [])
        xg_h, xg_a = 0.0, 0.0
        for clk, tid, val in cand_shots:
            if t0_c <= clk <= t1_c:
                if tid == ht:
                    xg_h += val
                elif tid == at:
                    xg_a += val
        s['xg_home'] = xg_h
        s['xg_away'] = xg_a
        s['is_home_pp'] = int(s['n_home_skaters'] > s['n_away_skaters'])
        s['is_away_pp'] = int(s['n_away_skaters'] > s['n_home_skaters'])
        
    test_stints_by_match[mid] = stints_list

total_test_stints = sum(len(st) for st in test_stints_by_match.values())
print(f"Сформировано {total_test_stints:,} отрезков постоянного состава.")

test_skater_ids = sorted(list(set(toi_test['player_id'])))
print(f"Расчет регулярного RAPM (Ridge L2) с блочным бутстрэпом (30 итераций)...")

df_ratings = rapm.fit_rapm_with_bootstrap(
    test_stints_by_match,
    test_skater_ids,
    n_bootstraps=30,
    lambda_reg=80.0
)

# Метаданные
p_meta = all_players[['player_id', 'position', 'shoots']].drop_duplicates('player_id')
df_ratings = df_ratings.merge(p_meta, on='player_id', how='left')

p_toi_summary = toi_test.groupby('player_id').agg(
    matches_played=('match_id', 'nunique'),
    toi_total_min=('toi_min', 'sum')
).reset_index()
p_toi_summary['toi_per_game'] = p_toi_summary['toi_total_min'] / p_toi_summary['matches_played']
df_ratings = df_ratings.merge(p_toi_summary, on='player_id', how='left')

p_teams = player_seasons_test[['player_id', 'team_id']].drop_duplicates('player_id')
df_ratings = df_ratings.merge(p_teams, on='player_id', how='left')

df_ratings = df_ratings.sort_values('rating', ascending=False).reset_index(drop=True)
print("\nТоп-10 полевых игроков по Net Rating за 60 минут:")
print(df_ratings[['player_id', 'position', 'team_id', 'matches_played', 'rating', 'ci_lower', 'ci_upper', 'rating_off', 'rating_def']].head(10))

In [ ]:
# Добавление вратарей (базовый уровень 0.0) и экспорт test_ratings.csv
all_test_players = sorted(list(set(player_seasons_test['player_id'])))
existing_pids = set(df_ratings['player_id'])
missing_goalies = [p for p in all_test_players if p not in existing_pids]

if missing_goalies:
    goalie_rows = []
    for g_id in missing_goalies:
        pos_val = all_players.loc[all_players.player_id == g_id, 'position'].values
        pos = pos_val[0] if len(pos_val) > 0 and pd.notna(pos_val[0]) else 'G'
        t_val = player_seasons_test.loc[player_seasons_test.player_id == g_id, 'team_id'].values
        tid = t_val[0] if len(t_val) > 0 else 'UNKNOWN'
        goalie_rows.append({
            'player_id': g_id,
            'rating': 0.0,
            'ci_lower': -0.15,
            'ci_upper': 0.15,
            'rating_off': 0.0,
            'rating_def': 0.0,
            'position': pos,
            'shoots': np.nan,
            'matches_played': 0,
            'toi_total_min': 0.0,
            'toi_per_game': 0.0,
            'team_id': tid
        })
    df_ratings = pd.concat([df_ratings, pd.DataFrame(goalie_rows)], ignore_index=True)

export_cols = ['player_id', 'rating', 'ci_lower', 'ci_upper', 'position', 'team_id', 'matches_played', 'toi_total_min', 'rating_off', 'rating_def']
df_ratings[export_cols].to_csv('test_ratings.csv', index=False)

print(f"Файл test_ratings.csv успешно сформирован! Строк: {len(df_ratings)}, колонок: {len(export_cols)}")

In [ ]:
# Проверка устойчивости (Split-Half Reliability)
print("Сравнение устойчивости методик (Split-Half Reliability):")

val_m = set(games_train['match_id'].head(350))
skaters_sub = shifts_train[shifts_train.match_id.isin(val_m) & (~shifts_train.is_goalie)].copy()
events_sub = events_train[events_train.match_id.isin(val_m)].copy()
shots_sub = shots_train[shots_train.match_id.isin(val_m)].copy()
toi_sub = toi_train[toi_train.match_id.isin(val_m)].copy()

# Честный on-ice +/-
goals = events_sub[events_sub.event_type == 'goal'][['match_id', 'period', 'clock_s', 'actor_team_id']]
iv_grouped = {k: g for k, g in skaters_sub.groupby(['match_id', 'period'])}

pm_records = []
for r in goals.itertuples(index=False):
    mid, per, clk, g_team = r
    if (mid, per) not in iv_grouped: continue
    active = iv_grouped[(mid, per)]
    on_ice = active[(active.start_clock <= clk) & (active.end_clock >= clk)]
    for row in on_ice.itertuples(index=False):
        pm_records.append((mid, row.player_id, 1 if row.team_id == g_team else -1))
df_pm = pd.DataFrame(pm_records, columns=['match_id', 'player_id', 'pm']).groupby(['match_id', 'player_id'])['pm'].sum().reset_index()

# Честный on-ice xG
xg_records = []
for r in shots_sub[['match_id', 'period', 'clock_s', 'actor_team_id', 'xg']].itertuples(index=False):
    mid, per, clk, s_team, val = r
    if (mid, per) not in iv_grouped: continue
    active = iv_grouped[(mid, per)]
    on_ice = active[(active.start_clock <= clk) & (active.end_clock >= clk)]
    for row in on_ice.itertuples(index=False):
        xg_records.append((mid, row.player_id, val if row.team_id == s_team else -val))
df_xg_onice = pd.DataFrame(xg_records, columns=['match_id', 'player_id', 'xg_net']).groupby(['match_id', 'player_id'])['xg_net'].sum().reset_index()

# Очки игрока (голы + передачи)
pts_ev = events_sub[events_sub.event_type.isin(['goal', 'assist_primary', 'assist_secondary'])]
p_goals = pts_ev[pts_ev.event_type == 'goal'].groupby(['match_id', 'actor_id']).size().rename('goals')
p_assists = pts_ev[pts_ev.event_type.str.startswith('assist_')].groupby(['match_id', 'actor_id']).size().rename('assists')
df_pts = pd.concat([p_goals, p_assists], axis=1).fillna(0).reset_index().rename(columns={'actor_id': 'player_id'})

# Действия (передачи, входы, отборы)
act_ev = events_sub[events_sub.event_type.isin(['pass_complete', 'key_pass_complete', 'steal', 'entry_success', 'exit_success'])]
df_acts = act_ev.groupby(['match_id', 'actor_id']).size().rename('action_val').reset_index().rename(columns={'actor_id': 'player_id'})

m_stats = toi_sub.merge(df_pm, on=['match_id', 'player_id'], how='left').fillna({'pm': 0})
m_stats = m_stats.merge(df_xg_onice, on=['match_id', 'player_id'], how='left').fillna({'xg_net': 0})
m_stats = m_stats.merge(df_pts, on=['match_id', 'player_id'], how='left').fillna({'goals': 0, 'assists': 0})
m_stats = m_stats.merge(df_acts, on=['match_id', 'player_id'], how='left').fillna({'action_val': 0})
m_stats['plus_minus'] = m_stats['pm']

split_half_df = ev.evaluate_split_half_reliability(m_stats, min_matches=4)
print(split_half_df)

In [ ]:
# Анализ состава ХК «Авангард» (T113)
avangard_ratings = df_ratings[df_ratings.team_id == 'T113'].sort_values('rating', ascending=False)
print(f"Игроков «Авангарда» оценено: {len(avangard_ratings)}")
print("\nТоп-10 игроков «Авангарда» по комплексному рейтингу:")
print(avangard_ratings[['player_id', 'position', 'matches_played', 'toi_per_game', 'rating', 'ci_lower', 'ci_upper', 'rating_off', 'rating_def']].head(10).to_string())